In [1]:
import os
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 1. Load the Kaggle dataset
data_path = '../input/Credit Card Fraud Detection/creditcard.csv'

if not os.path.exists(data_path):
    for root, dirs, files in os.walk('../input'):
        for file in files:
            if file == 'creditcard.csv':
                data_path = os.path.join(root, file)
                break

print(f"Loading dataset from: {data_path}")
df = pd.read_csv(data_path)

# 2. Preprocess Data
X_raw = df.drop(columns=['Time', 'Class']).values
y = df['Class'].values.reshape(-1, 1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

X = np.c_[np.ones((X_scaled.shape[0], 1)), X_scaled]

print(f"Dataset Shape: {X.shape[0]} samples, {X.shape[1] - 1} features (+1 intercept term)")

# 3. Logistic Regression Helper Functions
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def compute_loss(X, y, theta):
    m = len(y)
    h = sigmoid(X.dot(theta))
    loss = -(1 / m) * np.sum(y * np.log(h + 1e-15) + (1 - y) * np.log(1 - h + 1e-15))
    return loss


def batch_gradient_descent(X, y, lr=0.1, epochs=30):
    m, n = X.shape
    theta = np.zeros((n, 1))
    updates = 0
    
    start_time = time.time()
    for epoch in range(epochs):
        h = sigmoid(X.dot(theta))
        gradient = (1 / m) * X.T.dot(h - y)
        theta -= lr * gradient
        updates += 1
        
    execution_time = time.time() - start_time
    final_loss = compute_loss(X, y, theta)
    return execution_time, updates, final_loss

def stochastic_gradient_descent(X, y, lr=0.005, epochs=1):
    m, n = X.shape
    theta = np.zeros((n, 1))
    updates = 0
    
    start_time = time.time()
    for epoch in range(epochs):
        indices = np.random.permutation(m)
        X_shuffled = X[indices]
        y_shuffled = y[indices]
        
        for i in range(m):
            xi = X_shuffled[i : i+1]
            yi = y_shuffled[i : i+1]
            h = sigmoid(xi.dot(theta))
            gradient = xi.T.dot(h - yi)
            theta -= lr * gradient
            updates += 1
            
    execution_time = time.time() - start_time
    final_loss = compute_loss(X, y, theta)
    return execution_time, updates, final_loss

def mini_batch_gradient_descent(X, y, batch_size=64, lr=0.05, epochs=1):
    m, n = X.shape
    theta = np.zeros((n, 1))
    updates = 0
    
    start_time = time.time()
    for epoch in range(epochs):
        indices = np.random.permutation(m)
        X_shuffled = X[indices]
        y_shuffled = y[indices]
        
        for i in range(0, m, batch_size):
            xi = X_shuffled[i : i + batch_size]
            yi = y_shuffled[i : i + batch_size]
            
            h = sigmoid(xi.dot(theta))
            gradient = (1 / len(xi)) * xi.T.dot(h - yi)
            theta -= lr * gradient
            updates += 1
            
    execution_time = time.time() - start_time
    final_loss = compute_loss(X, y, theta)
    return execution_time, updates, final_loss

# Benchmarking Runners

print("\nExecuting Batch Gradient Descent...")
bgd_time, bgd_updates, bgd_loss = batch_gradient_descent(X, y, lr=0.1, epochs=30)

print("Executing Stochastic Gradient Descent...")
sgd_time, sgd_updates, sgd_loss = stochastic_gradient_descent(X, y, lr=0.005, epochs=1)

print("Executing Mini-Batch Gradient Descent (Size = 64)...")
mbgd_time, mbgd_updates, mbgd_loss = mini_batch_gradient_descent(X, y, batch_size=64, lr=0.05, epochs=1)

# Display Results 
print("\n" + "="*75)
print(f"{'Optimizer Approach':<30}{'Time (s)':<15}{'Weight Updates':<18}{'Final Loss':<12}")
print("="*75)
print(f"{'Batch GD':<30}{bgd_time:<15.4f}{bgd_updates:<18}{bgd_loss:<12.5f}")
print(f"{'Stochastic GD (SGD)':<30}{sgd_time:<15.4f}{sgd_updates:<18}{sgd_loss:<12.5f}")
print(f"{'Mini-Batch GD (size=64)':<30}{mbgd_time:<15.4f}{mbgd_updates:<18}{mbgd_loss:<12.5f}")
print("="*75)

Loading dataset from: ../input/datasets/organizations/mlg-ulb/creditcardfraud/creditcard.csv
Dataset Shape: 284807 samples, 29 features (+1 intercept term)

Executing Batch Gradient Descent...
Executing Stochastic Gradient Descent...
Executing Mini-Batch Gradient Descent (Size = 64)...

Optimizer Approach            Time (s)       Weight Updates    Final Loss  
Batch GD                      0.4332         30                0.29310     
Stochastic GD (SGD)           4.4240         284807            0.00473     
Mini-Batch GD (size=64)       0.2473         4451              0.00796     


In [2]:
import os
import time
import zipfile
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. HANDLE AND EXTRACT THE ZIPPED DATASET

zip_path = '../input/nyc-taxi-trip-duration/train.zip'
extracted_csv = './train.csv'

if not os.path.exists(zip_path):
    for root, dirs, files in os.walk('../input'):
        for file in files:
            if file == 'train.zip':
                zip_path = os.path.join(root, file)
                break

print(f"Extracting {zip_path}...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('.')
print("Extraction complete!")

print(f"Loading NYC Taxi dataset from: {extracted_csv}")
df = pd.read_csv(extracted_csv, nrows=500000) 

# 2. FEATURE EXTRACTION & PREPROCESSING
feature_cols = ['passenger_count', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude']
X_raw = df[feature_cols].values
y = df['trip_duration'].values.reshape(-1, 1)

valid_indices = (y > 10) & (y < 86400)  # trips between 10 seconds and 1 day
X_raw = X_raw[valid_indices.flatten()]
y = y[valid_indices.flatten()]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

X = np.c_[np.ones((X_scaled.shape[0], 1)), X_scaled]
print(f"Cleaned Dataset Shape: {X.shape[0]} trips, {X.shape[1] - 1} features (+1 bias)\n")

# 3. COST FUNCTION & GRADIENT DESCENT IMPLEMENTATIONS

def compute_loss(X, y, theta):
    m = len(y)
    h = X.dot(theta)  
    error = h - y
    loss = (1 / (2 * m)) * np.sum(error ** 2)
    return loss

def batch_gradient_descent(X, y, lr=0.1, epochs=30):
    m, n = X.shape
    theta = np.zeros((n, 1))
    updates = 0
    
    start_time = time.time()
    for epoch in range(epochs):
        h = X.dot(theta)
        gradient = (1 / m) * X.T.dot(h - y)
        theta -= lr * gradient
        updates += 1
        
    execution_time = time.time() - start_time
    final_loss = compute_loss(X, y, theta)
    return execution_time, updates, final_loss

def stochastic_gradient_descent(X, y, lr=0.001, epochs=1):
    m, n = X.shape
    theta = np.zeros((n, 1))
    updates = 0
    
    start_time = time.time()
    for epoch in range(epochs):
        indices = np.random.permutation(m)
        X_shuffled = X[indices]
        y_shuffled = y[indices]
        
        for i in range(m):
            xi = X_shuffled[i : i+1]
            yi = y_shuffled[i : i+1]
            h = xi.dot(theta)
            gradient = xi.T.dot(h - yi)
            theta -= lr * gradient
            updates += 1
            
    execution_time = time.time() - start_time
    final_loss = compute_loss(X, y, theta)
    return execution_time, updates, final_loss

def mini_batch_gradient_descent(X, y, batch_size=64, lr=0.05, epochs=1):
    m, n = X.shape
    theta = np.zeros((n, 1))
    updates = 0
    
    start_time = time.time()
    for epoch in range(epochs):
        indices = np.random.permutation(m)
        X_shuffled = X[indices]
        y_shuffled = y[indices]
        
        for i in range(0, m, batch_size):
            xi = X_shuffled[i : i + batch_size]
            yi = y_shuffled[i : i + batch_size]
            
            h = xi.dot(theta)
            gradient = (1 / len(xi)) * xi.T.dot(h - yi)
            theta -= lr * gradient
            updates += 1
            
    execution_time = time.time() - start_time
    final_loss = compute_loss(X, y, theta)
    return execution_time, updates, final_loss

# 4. RUN BENCHMARKS
print("Running Batch Gradient Descent...")
bgd_time, bgd_updates, bgd_loss = batch_gradient_descent(X, y, lr=0.1, epochs=30)

print("Running Stochastic Gradient Descent (This might take a few seconds)...")
sgd_time, sgd_updates, sgd_loss = stochastic_gradient_descent(X, y, lr=0.001, epochs=1)

print("Running Mini-Batch Gradient Descent (size=64)...")
mbgd_time, mbgd_updates, mbgd_loss = mini_batch_gradient_descent(X, y, batch_size=64, lr=0.05, epochs=1)

# 5. OUTPUT SUMMARY RESULTS
print("\n" + "="*75)
print(f"{'Optimizer Approach':<30}{'Time (s)':<15}{'Weight Updates':<18}{'Final MSE Loss':<12}")
print("="*75)
print(f"{'Batch GD':<30}{bgd_time:<15.4f}{bgd_updates:<18}{bgd_loss:<12.2f}")
print(f"{'Stochastic GD (SGD)':<30}{sgd_time:<15.4f}{sgd_updates:<18}{sgd_loss:<12.2f}")
print(f"{'Mini-Batch GD (size=64)':<30}{mbgd_time:<15.4f}{mbgd_updates:<18}{mbgd_loss:<12.2f}")
print("="*75)

Extracting ../input/competitions/nyc-taxi-trip-duration/train.zip...
Extraction complete!
Loading NYC Taxi dataset from: ./train.csv
Cleaned Dataset Shape: 499263 trips, 5 features (+1 bias)

Running Batch Gradient Descent...
Running Stochastic Gradient Descent (This might take a few seconds)...
Running Mini-Batch Gradient Descent (size=64)...

Optimizer Approach            Time (s)       Weight Updates    Final MSE Loss
Batch GD                      0.1952         30                4939072.19  
Stochastic GD (SGD)           3.2102         499263            4956534.97  
Mini-Batch GD (size=64)       0.1827         7801              4943427.81  
